# Random Survival Forest

> **Prerequisite:** Run `clean_data.ipynb` first. This notebook reads from `cleaned_data/`.

Trains a Random Survival Forest on `mm_nopgs` and `mm_pgs_score`, reports 5-fold CV C-index.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GridSearchCV
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

## Load data

In [2]:
for p in ['cleaned_data/mm_pgs_score.csv', 'cleaned_data/mm_nopgs.csv']:
    assert Path(p).exists(), f'{p} not found — run clean_data.ipynb first.'

df_pgs   = pd.read_csv('cleaned_data/mm_pgs_score.csv', index_col='ID')
df_nopgs = pd.read_csv('cleaned_data/mm_nopgs.csv',     index_col='ID')

print(f'PGS dataset:    {len(df_pgs)} rows,   columns: {list(df_pgs.columns)}')
print(f'No-PGS dataset: {len(df_nopgs)} rows, columns: {list(df_nopgs.columns)}')

PGS dataset:    2000 rows,   columns: ['ancestry', 'age', 'm_spike', 'sflc_ratio', 'creatinine', 'pgs_score', 'status', 'time_years']
No-PGS dataset: 2000 rows, columns: ['ancestry', 'age', 'm_spike', 'sflc_ratio', 'creatinine', 'status', 'time_years']


## Train & evaluate

This takes around 15min on my Lenovo IdeaPad.

In [3]:
features_nopgs = ['age', 'm_spike', 'sflc_ratio', 'creatinine']
features_pgs   = ['age', 'm_spike', 'sflc_ratio', 'creatinine', 'pgs_score']

y_nopgs = Surv.from_dataframe('status', 'time_years', df_nopgs)
y_pgs   = Surv.from_dataframe('status', 'time_years', df_pgs)

estimator_counts = [20, 50, 100, 150, 200]
param_grid = {'n_estimators': estimator_counts}

rsf = RandomSurvivalForest(min_samples_leaf=10, random_state=42)

gs_nopgs = GridSearchCV(rsf, param_grid, cv=5, refit=False, n_jobs=len(estimator_counts))
gs_nopgs.fit(df_nopgs[features_nopgs], y_nopgs)

gs_pgs = GridSearchCV(rsf, param_grid, cv=5, refit=False, n_jobs=len(estimator_counts))
gs_pgs.fit(df_pgs[features_pgs], y_pgs)

results_nopgs = pd.DataFrame(gs_nopgs.cv_results_)[['param_n_estimators', 'mean_test_score', 'std_test_score']]
results_pgs   = pd.DataFrame(gs_pgs.cv_results_)[['param_n_estimators', 'mean_test_score', 'std_test_score']]

print('mm_nopgs:')
print(results_nopgs.to_string(index=False))
print()
print('mm_pgs_score:')
print(results_pgs.to_string(index=False))

mm_nopgs:
 param_n_estimators  mean_test_score  std_test_score
                 20         0.668789        0.015814
                 50         0.678882        0.016334
                100         0.677411        0.018579
                150         0.678096        0.019278
                200         0.679203        0.018380

mm_pgs_score:
 param_n_estimators  mean_test_score  std_test_score
                 20         0.687112        0.017611
                 50         0.684897        0.018684
                100         0.688921        0.016785
                150         0.691733        0.017918
                200         0.691863        0.016910
